# Pharmakaksha · BP701T · Unit IV B · ANOVA, Non-parametric and Chi-square Tests
**Learn · Grow · Achieve**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit4B_ANOVA_Nonparametric_and_Chi_Square.ipynb) &nbsp; [Open on learn.pharmakaksha.com](https://learn.pharmakaksha.com/notebooks/index.html?path=BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit4B_ANOVA_Nonparametric_and_Chi_Square.ipynb)

Companion notebook for BP701T Unit IV B study notes, slides and video. Every worked example from the unit is here, in order, followed by lab exercises and solutions.

> **How to use this notebook:** BP701T is a hand-calculation course. Solve each example with your calculator first, using the study notes, then run the cell to check your answer. In the exam you will not have Python.

## Using learn.pharmakaksha.com
- **First run is slow:** Python loads in your browser, which takes about 10 seconds the first time.
- **Kernel indicator:** the circle at the top right is filled while a cell runs and empty when it is idle.
- **Restart:** Kernel → Restart Kernel if something gets stuck, then run the cells again from the top.
- **Saving:** your changes are saved only in this browser. Download the notebook (File → Download) to keep a copy.
- **Start over:** delete your copy in the file browser to get a fresh notebook from the site.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("Ready")

## 4.7 One-way ANOVA
Disintegration time (min) of three brands of paracetamol tablets, 5 tablets each. Hand answers: SSB = 43.33, SSW = 30, F = 8.67 > F(0.05; 2, 12) = 3.89.

In [ ]:
bA = np.array([6, 8, 7, 5, 9]); bB = np.array([10, 12, 11, 9, 13]); bC = np.array([8, 9, 7, 10, 6])
groups = [bA, bB, bC]; allv = np.concatenate(groups); gm = allv.mean(); k = len(groups); N = len(allv)
ssb = sum(len(g) * (g.mean() - gm)**2 for g in groups)
ssw = sum(((g - g.mean())**2).sum() for g in groups)
msb, msw = ssb / (k - 1), ssw / (N - k)
print(f"means {[g.mean() for g in groups]}, grand mean {gm:.3f}")
print(f"SSB {ssb:.2f} | SSW {ssw:.2f} | MSB {msb:.2f} | MSW {msw:.2f} | F {msb/msw:.2f} | F table {stats.f.ppf(0.95, k-1, N-k):.2f}")
print(stats.f_oneway(bA, bB, bC))

### Which brands differ? Least significant difference (LSD)
LSD = t(0.05, 12) × √(2 × MSW / n) = 2.179 × 1 = 2.18. Differences: B − A = 4, B − C = 3, C − A = 1.

In [ ]:
lsd = stats.t.ppf(0.975, N - k) * np.sqrt(2 * msw / 5)
print(f"LSD = {lsd:.3f}")
for (n1, g1), (n2, g2) in [(("B", bB), ("A", bA)), (("B", bB), ("C", bC)), (("C", bC), ("A", bA))]:
    diff = g1.mean() - g2.mean(); print(f"{n1} - {n2} = {diff:.1f} ->", "significant" if abs(diff) > lsd else "not significant")
print(stats.tukey_hsd(bA, bB, bC))
plt.boxplot([bA, bB, bC], tick_labels=["Brand A", "Brand B", "Brand C"])
plt.ylabel("disintegration time (min)"); plt.title("One-way ANOVA: F = 8.67, p = 0.005"); plt.show()

## 4.8 Two-way ANOVA (without replication)
Hardness (kg/cm²) of three formulations (columns) made on four days (rows, the blocks). Hand answers: CF = 432, SST = 18, SS(formulations) = 8, SS(days) = 8.67, SSE = 1.33; F(form) = 18.0 > 5.14, F(days) = 13.0 > 4.76.

In [ ]:
data = np.array([[5, 7, 6], [6, 8, 8], [4, 6, 5], [5, 7, 5]])
r, c = data.shape; G = data.sum(); CF = G**2 / data.size
SST = (data**2).sum() - CF
SSC = (data.sum(axis=0)**2).sum() / r - CF
SSR = (data.sum(axis=1)**2).sum() / c - CF
SSE = SST - SSC - SSR
dfc, dfr, dfe = c - 1, r - 1, (c - 1) * (r - 1)
print(f"G {G} | CF {CF} | SST {SST} | SS form {SSC} | SS days {SSR:.2f} | SSE {SSE:.2f}")
print(f"F formulations = {(SSC/dfc)/(SSE/dfe):.2f} (table {stats.f.ppf(0.95, dfc, dfe):.2f})")
print(f"F days         = {(SSR/dfr)/(SSE/dfe):.2f} (table {stats.f.ppf(0.95, dfr, dfe):.2f})")
import statsmodels.api as sm
from statsmodels.formula.api import ols
df = pd.DataFrame([(f"D{i+1}", f"F{j+1}", data[i, j]) for i in range(r) for j in range(c)], columns=["day", "form", "hardness"])
print(sm.stats.anova_lm(ols("hardness ~ C(form) + C(day)", df).fit()))

## 4.9 Non-parametric tests
### Mann–Whitney U (two independent groups)
Pain-relief scores, drug vs placebo, 6 patients each. Hand answers: R₁ = 53, R₂ = 25, U = 4 ≤ table U = 5 → significant.

In [ ]:
drug = np.array([12, 15, 9, 14, 11, 16]); plac = np.array([8, 10, 6, 13, 5, 7])
ranks = stats.rankdata(np.concatenate([drug, plac])); R1 = ranks[:6].sum(); R2 = ranks[6:].sum()
U1 = 6*6 + 6*7/2 - R1; U2 = 6*6 - U1
print(f"R1 {R1}, R2 {R2}, U1 {U1}, U2 {U2}, U = {min(U1, U2)}")
print(stats.mannwhitneyu(drug, plac, method="exact"))

### Wilcoxon signed-rank (paired)
Pain scores of 8 patients before and after a drug. Hand answers: T⁺ = 35, T⁻ = 1, T = 1 ≤ table T = 3 → significant.

In [ ]:
before = np.array([8, 6, 9, 4, 7, 6, 9, 10]); after = np.array([4, 4, 3, 5, 2, 3, 2, 2])
d = before - after; rk = stats.rankdata(np.abs(d))
print("d =", d, "| ranks of |d| =", rk)
print("T+ =", rk[d > 0].sum(), "| T- =", rk[d < 0].sum())
print(stats.wilcoxon(before, after, method="exact"))

### Kruskal–Wallis (three or more independent groups)
Days to wound healing with three ointments. Hand answers: R = 35, 60, 25; H = 6.5 > χ²(0.05, 2) = 5.99 → significant.

In [ ]:
oA = [7, 8, 9, 10, 11]; oB = [12, 13, 14, 15, 16]; oC = [3, 4, 5, 6, 17]
rk = stats.rankdata(oA + oB + oC); R = [rk[0:5].sum(), rk[5:10].sum(), rk[10:15].sum()]
H = 12 / (15 * 16) * sum(x**2 / 5 for x in R) - 3 * 16
print("rank sums", R, "| H =", H, "| table", round(stats.chi2.ppf(0.95, 2), 3))
print(stats.kruskal(oA, oB, oC))

### Friedman (three or more related treatments)
Pain relief with three drugs P, Q, R in the same 6 patients (crossover). Hand answers: rank sums 7, 11, 18; χ²F = 10.33 > 5.99 → significant.

In [ ]:
X = np.array([[3, 5, 8], [2, 4, 7], [4, 6, 9], [1, 3, 6], [3, 5, 6], [5, 4, 8]])
Rk = np.apply_along_axis(stats.rankdata, 1, X); Rs = Rk.sum(axis=0); n, k = X.shape
chiF = 12 / (n * k * (k + 1)) * (Rs**2).sum() - 3 * n * (k + 1)
print("rank sums", Rs, "| chi2_F =", round(chiF, 2))
print(stats.friedmanchisquare(*X.T))

## 4.10 Chi-square tests
### Goodness of fit
Defective tablets found on four compression machines (total 120; equal expected = 30 each). Hand answer: χ² = 11.67 > 7.81 (df 3).

In [ ]:
obs = np.array([30, 20, 25, 45]); exp = np.full(4, obs.sum() / 4)
print("chi2 =", ((obs - exp)**2 / exp).sum(), "| table", round(stats.chi2.ppf(0.95, 3), 3))
print(stats.chisquare(obs))

### 2 × 2 test of independence
ADRs: drug 30 of 100, placebo 15 of 100. Hand answers: χ² = 6.45 (5.62 with Yates' correction) > 3.84.

In [ ]:
t = np.array([[30, 70], [15, 85]])
a, b = t[0]; c, d = t[1]; N = t.sum()
print("shortcut chi2 =", round(N * (a*d - b*c)**2 / ((a+b)*(c+d)*(a+c)*(b+d)), 3))
chi, p, dof, expd = stats.chi2_contingency(t, correction=False)
print("chi2", round(chi, 3), "p", round(p, 4), "\nexpected\n", expd)
print("Yates:", round(stats.chi2_contingency(t, correction=True)[0], 3))

## Worked example · A three-arm analgesic trial
Responders: placebo 8/20, 50 mg 12/20, 100 mg 16/20. Hand answers: expected 12 responders and 8 non-responders per arm; χ² = 6.67 > 5.99 (df 2). Three separate tests would raise the false-alarm risk to 1 − 0.95³ = 14%.

In [ ]:
tab = pd.DataFrame({"responder": [8, 12, 16], "non-responder": [12, 8, 4]}, index=["placebo", "50 mg", "100 mg"])
chi, p, dof, expd = stats.chi2_contingency(tab)
print(tab, "\n\nexpected:\n", expd, f"\n\nchi2 = {chi:.3f}, df = {dof}, p = {p:.4f}, table = {stats.chi2.ppf(0.95, dof):.3f}")
print("Family-wise error with 3 tests:", round(1 - 0.95**3, 3))
(tab["responder"] / 20 * 100).plot(kind="bar", color=["#9AA5A8", "#1F8A70", "#17725C"], rot=0)
plt.ylabel("% responders"); plt.title("Response rises with dose (χ² = 6.67, p = 0.036)"); plt.show()

## Lab exercises
1. Assay (%) of a drug by three methods, 4 samples each: M1 98, 100, 99, 101; M2 102, 104, 103, 103; M3 99, 98, 100, 99. Run one-way ANOVA.
2. Sedation scores (ordinal) in two groups: A 3, 5, 4, 6, 7; B 1, 2, 4, 2, 3. Apply the Mann–Whitney U test.
3. In 150 patients, 60 of 90 on drug X improved and 25 of 60 on drug Y improved. Test with χ².
4. Four analysts each assay three batches (rows = analysts): (98, 101, 99), (97, 100, 98), (99, 102, 100), (98, 100, 99). Two-way ANOVA.

### Solutions

In [ ]:
print("1:", stats.f_oneway([98, 100, 99, 101], [102, 104, 103, 103], [99, 98, 100, 99]))
print("2:", stats.mannwhitneyu([3, 5, 4, 6, 7], [1, 2, 4, 2, 3], method="exact"))
chi, p, *_ = stats.chi2_contingency([[60, 30], [25, 35]], correction=False); print(f"3: chi2 = {chi:.2f}, p = {p:.4f}")
y = np.array([[98, 101, 99], [97, 100, 98], [99, 102, 100], [98, 100, 99]])
dd = pd.DataFrame([(f"An{i}", f"B{j}", y[i, j]) for i in range(4) for j in range(3)], columns=["analyst", "batch", "assay"])
print("4:\n", sm.stats.anova_lm(ols("assay ~ C(batch) + C(analyst)", dd).fit()))